In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
    Created on Thu Jul 18 2025
    
    @author: Yaning
"""

import os
import torch
import pyro
from pyro.optim import Adam
import pyro.distributions as dist
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pickle
import seaborn as sns
import pickle
from pyro.infer import SVI, Trace_ELBO
import re
import pandas as pd

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
with open('array_cafe_gamble.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
with open('array_future.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
# get two contexts
data = data[0]

In [4]:
data = torch.tensor(data)

In [5]:
data = data.to(device)

In [6]:
data.shape

torch.Size([2, 30, 118, 6])

In [ ]:
# # for cafe gamble dataset
# data = data.reshape(60,170,8)

# for future cueing dataset
data = data.reshape(60,118,6)

In [8]:
def model(data):
    num_params = 4
    num_agents = data.shape[0]
    num_trials = data.shape[1]
    # define hyper priors over model parameters
    # prior over sigma of a Gaussian is a Gamma distribution
    a = pyro.param('a', torch.ones(num_params, device=device), constraint=dist.constraints.positive)
    lam = pyro.param('lam', torch.ones(num_params, device=device), constraint=dist.constraints.positive)
    tau = pyro.sample('tau', dist.Gamma(a, a/lam).to_event(1)) # mean = a / (a/lam) = lam

    sig = pyro.deterministic('sig', 1/torch.sqrt(tau)) # Gauss sigma

    # each model parameter has a hyperprior defining group level mean
    # in the form of a Normal distribution
    m = pyro.param('m', torch.zeros(num_params, device=device))
    s = pyro.param('s', torch.ones(num_params, device=device), constraint=dist.constraints.positive)
    mu = pyro.sample('mu', dist.Normal(m, s*sig).to_event(1)) # Gauss mu, wieso s*sig?

    # in order to implement groups, where each subject is independent of the others, pyro uses so-called plates.
    # you embed what should be done for each subject into the "with pyro.plate" context
    # the plate vectorizes subjects and adds an additional dimension onto all arrays/tensors
    # i.e. p1 below will have the length num_agents
    with pyro.plate('ag_idx', num_agents):
        # draw parameters from Normal and transform (for numeric trick reasons)
        # base_dist = dist.Normal(0., 1.).expand_by([num_params]).to_event(1)
        base_dist = dist.Normal(torch.zeros(num_params, device=device), torch.ones(num_params, device=device)).to_event(1)
        # Transform via the pointwise affine mapping y = loc + scale*x (-> Neal's funnel)
        transform = dist.transforms.AffineTransform(mu, sig)

        locs = pyro.sample('locs', dist.TransformedDistribution(base_dist, [transform]))
        # print(locs.shape)


    with pyro.plate('data', num_agents*num_trials):
        sigma_rate = torch.exp(locs[:,0]).unsqueeze(-1).expand(-1, num_trials)
        a = torch.exp(locs[:,1]).unsqueeze(-1).expand(-1, num_trials)
        b = torch.exp(locs[:,2]).unsqueeze(-1).expand(-1, num_trials)
        # tan_a = torch.exp(locs[:,1]).unsqueeze(-1).expand(-1, num_trials)
        # tan_b = locs[:,2].unsqueeze(-1).expand(-1, num_trials)
        # beta = torch.exp(locs[:,3]).unsqueeze(-1).expand(-1, num_trials)
        beta = torch.sigmoid(locs[:,3]).unsqueeze(-1).expand(-1, num_trials)*5
        # beta = torch.clamp(beta, min=1e-4, max=20.0)

        sigma_combine = sigma_rate/(1+b*torch.exp(-a*data[:,:,2]))
        
        # sigma_combine = sigma_rate
        # sigma_combine = sigma_rate*torch.log(data[:,:,2]*ln_a + 1)
        # sigma_combine = sigma_rate*(torch.tanh(tan_a*data[:,:,2] + tan_b) + 1)

        e_mean = (data[:,:,3])/(1 + sigma_combine**2)

        # log_num = beta * torch.log(e_mean + 1e-8)
        # log_den = torch.logsumexp(
        #     torch.stack([
        #         log_num,
        #         beta * torch.log(torch.tensor(20., device=device))
        #     ], dim=0),
        #     dim=0
        # )
        # p = torch.exp(log_num - log_den)

        sum = e_mean + torch.tensor(20., device=device)
        # p = torch.stack([(e_mean/sum)**beta, (torch.tensor(20., device=device)/sum)**beta])
        log1 = torch.log(e_mean)*beta
        log2 = torch.log(torch.tensor(20.))*beta
        p = torch.stack([torch.exp(log1), torch.exp(log2)])
        # p = torch.stack([(e_mean/sum)**beta, (torch.tensor(20., device=device)/sum)**beta])
        
        # p = torch.stack([(e_mean)**beta, (torch.tensor(20., device=device))**beta])
        # print(p.shape)
        p /= p.sum(dim=0)

        p = p[0]

        




    pyro.sample("obs", dist.Bernoulli(probs = p).to_event(2), obs=data[:,:,4])

    # return locs

In [9]:
def guide(data):
    num_params = 4
    num_agents = data.shape[0]
    # biject_to(constraint) looks up a bijective Transform from constraints.real 
    # to the given constraint. The returned transform is guaranteed to have 
    # .bijective = True and should implement .log_abs_det_jacobian().
    trns = torch.distributions.biject_to(dist.constraints.positive)

    # define mean vector and covariance matrix of multivariate normal
    m_hyp = pyro.param('m_hyp', torch.zeros(2*num_params, device=data.device))
    st_hyp = pyro.param('scale_tril_hyp',
                    torch.eye(2*num_params, device=data.device),
                    constraint=dist.constraints.lower_cholesky)

    # set hyperprior to be multivariate normal
    # scale_tril (Tensor) – lower-triangular factor of covariance, with positive-valued diagonal
    hyp = pyro.sample('hyp',
                    dist.MultivariateNormal(m_hyp, scale_tril=st_hyp),
                    infer={'is_auxiliary': True})

    # mu & tau unconstrained
    unc_mu = hyp[..., :num_params]
    unc_tau = hyp[..., num_params:]

    # constrained tau, shape num_params, or num_particles, 1, num_params
    c_tau = trns(unc_tau)

    # ld = log_density
    # log_abs_det_jacobian(x, y) computes derivative |dy/dx|
    ld_tau = -trns.inv.log_abs_det_jacobian(c_tau, unc_tau)
    
    # sum_rightmost(x, dim)
    # sum out ``dim`` many rightmost dimensions of a given tensor.
    # ld_tau.shape is num_params, or num_particles, 1, num_params before sum_rightmost
    ld_tau = dist.util.sum_rightmost(ld_tau, ld_tau.dim() - c_tau.dim() + 1)

    # some numerics tricks
    mu = pyro.sample("mu", dist.Delta(unc_mu, event_dim=1))
    # c_tau shape: num_params, or num_particles, 1, num_params
    # ld_tau shape: [] or num_particles, 1,
    tau = pyro.sample("tau", dist.Delta(c_tau, log_density=ld_tau, event_dim=1))

    m_locs = pyro.param('m_locs', torch.zeros(num_agents, num_params, device=data.device))
    st_locs = pyro.param('scale_tril_locs',
                    torch.eye(num_params, device=data.device).repeat(num_agents, 1, 1),
                    constraint=dist.constraints.lower_cholesky)
    
    with pyro.plate('ag_idx', num_agents):
        locs = pyro.sample("locs", dist.MultivariateNormal(m_locs, scale_tril=st_locs))

    return {'tau': tau, 'mu': mu, 'locs': locs, 'm_locs': m_locs, 'st_locs': st_locs}

In [ ]:
# this is for running the notebook in our testing framework
smoke_test = ('CI' in os.environ)
# the step was 2000
n_steps = 2 if smoke_test else 5000
# assert pyro.__version__.startswith('1.8.6')

# clear the param store in case we're in a REPL
pyro.clear_param_store()# setup the optimizer
# the learning rate was 0.0005 , "betas": (0.90, 0.999)
# tried "n_par":15 in adam params but it does not have this argument
adam_params = {"lr": 0.01}
optimizer = Adam(adam_params)
# setup the inference algorithm
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())
# svi = SVI(model_gamma, guide_gamma, optimizer, loss=Trace_ELBO())

loss = []
pbar = tqdm(range(n_steps), position = 0)
# do gradient steps
for step in pbar:
    loss.append(torch.tensor(svi.step(data)))
    pbar.set_description("Mean ELBO %6.2f" % torch.tensor(loss[-20:]).mean())
    # for name, value in pyro.get_param_store().items():
    #     print(name, pyro.param(name))
    if torch.isnan(loss[-1]):
	    break

plt.figure()
plt.plot(loss)
plt.xlabel("iter step")
plt.ylabel("ELBO loss")
plt.title("ELBO minimization during inference")
plt.show()

Get posterior parameters

In [146]:
pos_dict = {}
for name, value in pyro.get_param_store().items():
    pos_dict[name] = value

# change the dictionary to numpy instead of tensor
# because somehow the tensor cannot be save with pickle
numpy_dict = {key: value.detach().cpu().numpy() for key, value in pos_dict.items()}

Get samples from the guide

In [47]:
sample_num = 1000
tau = []
mu = []
locs = []
for i in range(sample_num):
    tau.append(guide(data)['tau'].detach().cpu().numpy())
    mu.append(guide(data)['mu'].detach().cpu().numpy())
    locs.append(guide(data)['locs'].detach().cpu().numpy())

Calculate likelihoods for WAIC

In [48]:
probs = []
for i in locs:
    sigma_rate = torch.tensor(i[:,0]).unsqueeze(-1).to(device)
    a = torch.tensor(i[:,1]).unsqueeze(-1).to(device)
    b = torch.tensor(i[:,2]).unsqueeze(-1).to(device)
    beta = torch.tensor(i[:,3]).unsqueeze(-1).to(device)

    sigma_rate = torch.exp(sigma_rate)
    a = torch.exp(a)
    b = torch.exp(b)
    beta = torch.exp(beta)

    sigma_combine = sigma_rate/(1+b*torch.exp(-a*data[:,:,2]))
    # sigma_combine = sigma_rate
    # sigma_combine = sigma_rate*(torch.tanh(a*data[:,:,2] + b) + 1)

    e_mean = (data[:,:,3])/(1 + sigma_combine**2)

    # e_mean = (data[:,:,3])/(sigma_combine + 1)
    sum = e_mean + torch.tensor(20., device=device)

    softmax_args = torch.stack([(e_mean/sum)**beta, (torch.tensor(20.)/sum)**beta])
    # softmax_args = torch.stack([beta*e_mean, beta*torch.tensor(1.)])
    softmax_args /= softmax_args.sum(dim=0)
    p = softmax_args[0]
    
    # softmax_args = torch.stack([beta*e_mean/sum, beta*torch.tensor(20., device=device)/sum])
    # p = torch.softmax(softmax_args, dim = 0)[0]
    probs.append(p)

In [50]:
likeli = []
for i in probs:
    temp = i*data[:,:,4] + (1-data[:,:,4])*(1-i)
    temp = temp.cpu()
    likeli.append(temp)

In [53]:
likeli = np.array(likeli)

In [55]:
# final = likeli.reshape(5100, 1000)
# final = likeli.reshape(10200, 1000)

final = likeli.reshape(60,170,1000)

In [58]:
final = torch.tensor(final)

In [59]:
torch.isnan(final).any()

tensor(False)

Single model WAIC value

In [79]:
log_likes = torch.log(final)

lppd = torch.logsumexp(log_likes, dim=-1) - torch.log(torch.tensor(1000))

print(lppd.shape)

lppd = lppd.sum(dim=1)

p_waic = log_likes.var(dim=1, unbiased=True).sum(dim=1)

waic = -2 * (lppd - p_waic)

torch.Size([60, 170])


In [80]:
waic

tensor([567.4138, 564.8978, 560.0452, 560.9593, 560.9087, 558.7243, 562.9635,
        559.7531, 560.8587, 565.0829, 568.2971, 564.9532, 573.3153, 572.2793,
        555.9020, 558.7334, 562.2743, 570.6697, 562.4424, 557.9249, 557.5877,
        562.5881, 570.6138, 557.5844, 565.9452, 557.2783, 557.7989, 559.2156,
        564.9948, 564.9919, 564.0430, 561.9546, 566.1503, 554.5391, 558.8830,
        564.9989, 557.5074, 562.9870, 563.6727, 554.2692, 571.0911, 558.7463,
        557.7824, 562.0220, 569.6483, 555.9735, 563.3293, 556.6958, 564.9550,
        576.3599, 564.5521, 559.0603, 555.8292, 573.1751, 564.5298, 570.8997,
        567.4732, 562.4504, 561.4219, 560.9093], dtype=torch.float64)

Compare

In [81]:
log_likes = torch.log(final)

lppd = torch.logsumexp(log_likes, dim=-1) - torch.log(torch.tensor(1000))
# lppd = lppd.sum(dim=0)

p_waic = log_likes.var(dim=-1, unbiased=True)

waic = -2 * (lppd - p_waic)

In [82]:
waic.shape

torch.Size([60, 170])

In [83]:
average = waic.sum(dim=1)

In [43]:
torch.sqrt(10200 * delta_waic.var(unbiased=True))

tensor(9.0334, dtype=torch.float64)